In [ ]:
#
# Project:
#      PyTorch Dojo (https://github.com/wo3kie/ml-dojo)
#
# Author:
#      Lukasz Czerwinski (https://www.lukaszczerwinski.pl/)
#

$$ \sigma(x) = \text{sigmoid}(x) = \frac{e^x}{1+e^x} $$

$$ f(x) = \tanh(x) = \frac{e^x - e^{-x}}{e^x + e^{-x}} =  2 \, \sigma(2x) - 1 $$

$$ \frac{\partial f}{\partial x} = 1 - f(x)^2 $$

$$ \diamond \diamond \diamond $$

$$ \tanh(x) \in (-1, 1) $$

$$ p(x) = \frac{1}{2} \Big( f(x) + 1 \Big) $$

$$ p(x) \in (0, 1) $$

$$ \frac{\partial p}{\partial x} = \frac{\partial p}{\partial f} \frac{\partial f}{\partial x} = \frac{1}{2} \Big(1 - f(x)^2\Big) $$

In [ ]:
import torch as tr
import torch.nn as nn
import torch.autograd as autograd

import import_ipynb
import sigmoid # type: ignore
from approx import approx # type: ignore


def tanh(x: tr.Tensor) -> tr.Tensor:
    """Hyperbolic tangent function."""
    
    y = 2 * sigmoid.sigmoid(2 * x) - 1
    return y
    

class TanhFunction(autograd.Function):
    """Custom autograd function with `forward` and `backward` passes for the `tanh`."""

    @staticmethod
    def forward(ctx, x: tr.Tensor) -> tr.Tensor:
        # For demonstration we save `x`, however saving `y` would be more efficient.
        ctx.save_for_backward(x)

        y = tanh(x)  
        return y

    @staticmethod
    def backward(ctx, grad_output: tr.Tensor) -> tuple[tr.Tensor, ]:
        (x, ) = ctx.saved_tensors

        df_dx = 1 - tanh(x) ** 2
        dF_dx = grad_output * df_dx
        assert dF_dx.shape == x.shape

        return (dF_dx, )
    

class Tanh(nn.Module):
    """Custom module for the `tanh`."""

    def forward(self, x: tr.Tensor) -> tr.Tensor:
        y = TanhFunction.apply(x)
        return y


In [ ]:
# Unit tests

def test_basic() -> None:
    assert tanh(1000) == approx(tr.tensor(1.0))
    assert tanh(-1000) == approx(tr.tensor(-1.0))
    assert tanh(0) == approx(tr.tensor(0.0))
    assert tanh(1) == approx(tr.tensor(0.76159))
    assert tanh(-1) == approx(tr.tensor(-0.76159))


def test_gradcheck(samples) -> None:
    def func(x: tr.Tensor) -> tr.Tensor:
        return TanhFunction.apply(x)

    x = tr.randn(samples, dtype=tr.float64, requires_grad=True)
    assert autograd.gradcheck(func, (x, ), eps=0.001, atol=0.001)


def test_compare(samples) -> None:
    x = tr.randn(samples, dtype=tr.float32, requires_grad=True)

    x1 = x.clone().detach().requires_grad_(True)
    y1 = Tanh()(x1)
    F1 = y1.sum()
    F1.backward()

    x2 = x.clone().detach().requires_grad_(True)
    y2 = tr.tanh(x2)
    F2 = y2.sum()
    F2.backward()

    assert y1 == approx(y2)
    assert x1.grad == approx(x2.grad)


if __name__ == "__main__":
    test_basic()

    test_gradcheck(1)
    test_gradcheck(10)
    test_gradcheck(100)
    
    test_compare(1)
    test_compare(10)
    test_compare(100)